# A2-Lite — Vehicle-only geometry teaching pilot
Latest revision: **A2-LITE-512-VEHICLE-KD-2026-10-14-r1**

Question: can we retain the Vehicle improvement while preserving Pedestrian accuracy and recall?

Only one objective change from the reviewed rebalanced run: **remove direct Pedestrian teacher contributions**. Vehicle guidance keeps its original per-component class share. If both classes qualified, Vehicle still contributes half of its class mean; it is NOT doubled when Pedestrian guidance is removed. If only Vehicle qualified for a component, it retains its existing full class share. Original eligibility masks determine this denominator on the unchanged full target list.

Teacher geometry weights remain size=1, depth=1, orientation=0.01, with the same overall ramp to 0.25. No direct classification/box teacher losses. Ground-truth labels and ALL label losses remain enabled for BOTH classes. This does not remove pedestrians from training, rebalance the dataset, freeze shared features, or guarantee pedestrian preservation.

Same A2 teacher, A2-Lite architecture, 640×192 input, 50 queries, original pruned weights, exact 512 TRAIN frames, five epochs, seed, optimizer, temperature 2, matching and teacher eligibility. Frame groups are balanced, object counts are not. Never initialize from the previous trained endpoint. Reuse the original teacher cache; no teacher inference or historical reference training is repeated.

Compare fixed epoch 5 on all 3,769 VAL images with label-only, all-head KD, original geometry-only and rebalanced geometry-only. Primary benefit guardrail versus label-only stays unchanged: all five AP metrics nonregressing, mean moderate 3D AP gain ≥0.1 point, nearby recall drop ≤2 absolute percentage points for either class. Relative benefit is NOT product/deployment qualification. Shared-feature effects remain possible; if this arm fails, stop this KD recipe and prioritize label-based student recovery rather than adding variants automatically.

A100 preferred. Keep the current balanced-run session if possible: its local inputs are the first staging candidate, avoiding repeated Drive reads. Use the completed reusable archive if available; otherwise four-worker verified staging. No native extension, old CUDA stack, Torch replacement, architecture change or Core ML conversion. No phone is needed.

Run sections 1–8. After a reset rerun 1–5, then resume 6. Checkpoints save every 128 updates; up to 127 interrupted updates may repeat. Return the ZIP after execution or quality failure. No full-data training follows automatically.

## 1. Drive, all paths and helpers
BALANCED can be the completed rebalanced folder or its results ZIP. PILOT remains the ORIGINAL 512-frame recovery folder/ZIP. Adjust these paths only if your Drive folders differ.


In [ ]:
# A2-LITE-512-VEHICLE-KD-2026-10-14-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys, zipfile
REVISION='A2-LITE-512-VEHICLE-KD-2026-10-14-r1'
RUN_ID='a2_lite_512_vehicle_kd_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID):raise ValueError('Use lowercase letters, digits and underscores')
MOBILE_REPO=Path('/content/mobile_adas3d')
BALANCED=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_balanced_geometry/a2_lite_512_angle001_kd_r1')
PILOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_recovery/a2_lite_640_recovery_kd_r1')
INITIALIZATION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1/a2_lite_initialization.pth')
COMPACT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution/a2_lite_resolution_cpu_screen_r1')
SPLITS=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_VEHICLE_'+RUN_ID)
DATASET=Path('/content/kitti_a2_lite_accuracy_vehicle_'+RUN_ID)
# Optional reusable input archive. Do not prepare it in this GPU run.
CACHE=Path('/content/drive/MyDrive/mobile_adas3d_inputs/kitti_chen_a2_v1')
LOCAL_ARCHIVE=Path('/content/a2_lite_archive_local/kitti_chen_7481.zip')
CANDIDATES=[Path(p) for p in (
    '/content/kitti_a2_lite_accuracy_angle001_a2_lite_512_angle001_kd_r1',
    '/content/kitti_a2_lite_accuracy_geometry_a2_lite_512_geometry_kd_r1',
    '/content/kitti_a2_lite_accuracy_recovery_a2_lite_640_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_full_a2_lite_640_full_recovery_kd_r1',
    '/content/kitti_a2_lite_accuracy_cache_prepare_v1',
    '/content/kitti_a2_lite_accuracy_a2_lite_640_accuracy_r1',
    '/content/kitti_m68','/content/kitti_m67','/content/kitti',
    '/content/monodetr_kitti_a2','/content/drive/MyDrive/datasets/kitti')]
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_vehicle_geometry')/RUN_ID
PYTHON=Path(sys.executable);SCRIPT=MOBILE_REPO/'scripts/recover_a2_lite_vehicle_geometry.py'
LOGS=OUTPUT/'colab_logs';LOGS.mkdir(parents=True,exist_ok=True)
READY=False;TINY_PASSED=False;KD_READY=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command));log_path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',log_path,flush=True);tail=deque(maxlen=40)
    with log_path.open('w') as log:
        log.write(shlex.join(command)+'\n');log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush();tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        finally:process.stdout.close()
        status=process.wait();log.write(f'\nEXIT CODE: {status}\n');log.flush()
    if status:raise RuntimeError(f'Exit {status}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
def stage(name,*extra):
    if not READY and name!='bundle':raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--output',OUTPUT,'--device','cuda',*extra],name,MOBILE_REPO)
print(REVISION,'\nTeacher: Vehicle only, no redistribution; depth=1, size=1, orientation=0.01. GT: both classes unchanged.\nOutput:',OUTPUT)


## 2. Update main and check the current runtime
Pull before checking helper files. Preserve old runs and manifests. Ordinary package versions must match the reviewed references; do not replace Torch/CUDA or restore an old virtual environment to force a comparison.


In [ ]:
READY=False
if not MOBILE_REPO.exists():run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty:raise RuntimeError('Preserve tracked edits and use clean main.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main':raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only'],'update',MOBILE_REPO)
if not SCRIPT.is_file():raise FileNotFoundError('Pull the published Vehicle-only teaching notebook and helpers from main')
for path in (BALANCED,PILOT,INITIALIZATION,COMPACT,SPLITS/'train.txt',SPLITS/'val.txt'):
    if not path.exists():raise FileNotFoundError(f'{path}: correct section 1 paths')
if PILOT.is_dir():saved=json.loads((PILOT/'recovery_manifest.json').read_text())['cpu_environment']
else:
    with zipfile.ZipFile(PILOT) as archive:
        names=[n for n in archive.namelist() if n=='recovery_manifest.json' or n.endswith('/recovery_manifest.json')]
        if len(names)!=1:raise RuntimeError('Expected one original 512-frame pilot manifest')
        saved=json.loads(archive.read(names[0]))['cpu_environment']
probe=("import sys,json,torch; saved=json.loads(sys.argv[1]); "
       "actual={'python':'.'.join(map(str,sys.version_info[:2])),'torch':torch.__version__.split('+')[0]}; "
       "changes={k:{'saved':saved[k],'current':v} for k,v in actual.items() if saved[k]!=v}; "
       "assert not changes,'Stop for fresh matched-reference review; do not restore old CUDA: '+json.dumps(changes)")
run_logged([PYTHON,'-c',probe,json.dumps(saved)],'reference_python_torch')
packages=[name+'=='+saved[name] for name in ('numpy','timm','onnxruntime','pillow','scipy','opencv-python-headless')]+['pyyaml','scikit-image']
run_logged([PYTHON,'-m','pip','install','-q',*packages],'ordinary_dependencies')
run_logged([PYTHON,'-c',"import torch;assert torch.cuda.is_available(),'Select a GPU runtime';torch.cuda.init();print(torch.__version__,torch.cuda.get_device_name(0),'portable training; no native attention extension')"],'gpu_probe')
READY=True


## 3. Verify the reviewed parent and stage efficiently
Check the exact completed rebalanced parent and embedded gradient evidence before staging. A completed input archive is restored once onto local disk and verified. If it is unavailable, four-worker staging checks the previous balanced run's local data first, then the other candidates. No bad/completed cache is overwritten.

Keep this runtime open instead of switching CPU/GPU sessions. After a reset local data are lost; the separate one-time CPU archive workflow is optional for later sessions. All input hashes and the original TRAIN/VAL splits are checked again by experiment preparation.


In [ ]:
if not READY:raise RuntimeError('Run sections 1 and 2 first')
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_vehicle_geometry.py','-v'],'vehicle_tests',MOBILE_REPO)
probe=("import sys;from pathlib import Path;sys.path.insert(0,sys.argv[1]); "
       "from recover_a2_lite_vehicle_geometry import reviewed_balanced; "
       "m,comparison,records=reviewed_balanced(Path(sys.argv[2])); "
       "print('Approved balanced result verified; original training subset:',len(m['pilot_ids']))")
run_logged([PYTHON,'-c',probe,MOBILE_REPO/'scripts',BALANCED],'review_parent_evidence',MOBILE_REPO)
extra=[]
for candidate in CANDIDATES:extra += ['--candidate',candidate]
if (CACHE/'kitti_input_cache.json').is_file():
    if not (CACHE/'kitti_chen_7481.zip').is_file():raise RuntimeError('Completed cache receipt exists but archive is missing; preserve cache and review')
    print('Using completed reusable input archive; no per-image Drive copying.')
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_input_cache.py','-v'],'input_cache_tests',MOBILE_REPO)
    run_logged([PYTHON,'-u',MOBILE_REPO/'scripts/cache_a2_lite_inputs.py','restore',
                '--pilot',PILOT,'--cache',CACHE,'--dataset',DATASET,'--local-archive',LOCAL_ARCHIVE],'restore_input_archive',MOBILE_REPO)
else:
    print('No completed archive; four-worker staging prefers the completed local balanced-run inputs.')
    run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_data_staging.py','-v'],'staging_tests',MOBILE_REPO)
    run_logged([PYTHON,'-u',MOBILE_REPO/'scripts/stage_a2_lite_data.py',
                '--pilot',PILOT,'--dataset',DATASET,'--workers','4',*extra],'parallel_staging',MOBILE_REPO)
stage('prepare','--balanced',BALANCED,'--pilot',PILOT,'--initialization',INITIALIZATION,
      '--compact',COMPACT,'--repo',SOURCE_REPO,'--dataset',DATASET,'--split-dir',SPLITS,*extra)
MANIFEST=json.loads((OUTPUT/'geometry_manifest.json').read_text())
assert MANIFEST['recipe']['teacher_class_weights']=={'Pedestrian':0.0,'Vehicle':1.0}
assert MANIFEST['recipe']['teacher_component_weights']=={'dimensions':1.0,'depth':1.0,'angle':0.01}
assert len(MANIFEST['pilot_ids'])==512 and MANIFEST['recipe']['epochs']==5
print('New manifest:',MANIFEST['revision'],'— fixed updates:',512*5)
print('Class averaging:',MANIFEST['recipe']['teacher_class_normalization'])
print('Ground-truth classes and weights unchanged:',MANIFEST['recipe']['gt_weights'])


## 4. Original initialization parity and label-only tiny fit
Check the original 640×192 fixtures and the unchanged disposable 16-frame, 128-update ground-truth learnability test. Both classes still learn in this test. Its checkpoint NEVER initializes this pilot. If it fails, stop and return the ZIP without lowering the gate.


In [ ]:
stage('smoke')
stage('tiny')
TINY=json.loads((OUTPUT/'tiny_fit.json').read_text());TINY_PASSED=bool(TINY['passed'])
print(json.dumps({k:TINY[k] for k in ('passed','gt_loss_ratio','geometry_error_ratio','geometry_ratio_per_class','evaluation_scope')},indent=2))
if not TINY_PASSED:print('STOP: return section 8 evidence; do not lower the gate.')


## 5. Reuse original teacher records and verify Vehicle-only gradients
Cache both classes' original predictions unchanged for provenance. The loss, not the cache, disables Pedestrian teacher contributions. Check finite nonzero Vehicle gradients for depth, size and orientation, zero Pedestrian teacher counts for ALL components, no direct classification/box teacher counts, and unchanged weights/buffers with zero optimizer steps.


In [ ]:
KD_READY=False
if TINY_PASSED:
    stage('cache','--pilot',PILOT)
    CACHE_REPORT=json.loads((OUTPUT/'teacher_cache_complete.json').read_text())
    print(json.dumps({k:CACHE_REPORT[k] for k in ('images','encoded_gt_counts','qualified_teacher_matches','kd_pilot_authorized','teacher_inference_performed')},indent=2))
    print('Both original teacher classes are cached for provenance; the objective applies only Vehicle contributions.')
    if CACHE_REPORT['kd_pilot_authorized']:
        stage('kd-smoke')
        CHECK=json.loads((OUTPUT/'vehicle_kd_smoke.json').read_text());KD_READY=bool(CHECK['passed'])
        assert CHECK['optimizer_steps']==0 and CHECK['weights_and_buffers_unchanged']
        assert CHECK['vehicle_weight_not_redistributed']
        assert all(v['Pedestrian']==0 for v in CHECK['active_counts'].values())
        print(json.dumps({k:CHECK[k] for k in ('passed','teacher_class_weights','teacher_component_weights',
                                             'active_counts','component_gradient_abs_sum','optimizer_steps')},indent=2))
    else:print('STOP: teacher cache coverage did not qualify; return section 8 evidence.')
else:print('Skipped: tiny learnability check failed.')


## 6. Train ONE new arm — five fixed epochs
2,560 updates, batch size 1, original pruned weights, unchanged deterministic order, BOTH ground-truth classes and losses enabled. Vehicle-only teacher contributions retain their original averaging factors and the same orientation multiplier 0.01. No Pedestrian teacher weight is reassigned to Vehicle.

Same learning rates, AdamW, clipping, frozen BatchNorm/early backbone, no augmentation/dropout/AMP. The new kd/ checkpoint belongs only to this arm. Do not change the recipe or RUN_ID midway. Rerun 1–5 after interruption, then resume here.


In [ ]:
if KD_READY:
    stage('train')
else:print('No Vehicle-only training started; return the preflight evidence.')


## 7. Evaluate fixed epoch 5 and compare five recipes
All 3,769 VAL frames, no best-epoch selection. Four historical endpoints remain read-only; only this arm is retrained. Higher Vehicle accuracy cannot hide a Pedestrian AP or recall failure. This single-seed historical comparison does not isolate all shared-feature or stochastic effects. If the primary label-only benefit guardrail fails, stop this KD recipe; no automatic additional variant or longer training.


In [ ]:
if KD_READY:
    stage('evaluate')
    stage('compare')
    REPORT=json.loads((OUTPUT/'vehicle_geometry_comparison.json').read_text())
    for name,value in REPORT['results'].items():
        print(name,json.dumps({'metrics':value['metrics'],
                              'nearby_recall':{c:value['nearby']['classes'][c]['near_recall'] for c in ('Vehicle','Pedestrian')}},indent=2))
    print(json.dumps({k:REPORT[k] for k in ('teacher_class_weights','vehicle_weight_not_redistributed','comparisons',
                                          'benefit_guardrail_passed','next_review','comparison_limit',
                                          'full_training_authorized','deployment_qualified')},indent=2))
else:print('Validation skipped: preflight did not qualify training.')


## 8. Download results — also after failure
Return a2_lite_vehicle_geometry_results.zip. It contains metrics, predictions, original-cache identities, reviewed parent/audit evidence and combined logs; excludes large weights, model packages and images. Checkpoints remain on Drive for resume. No phone test yet.


In [ ]:
BUNDLE=OUTPUT/'a2_lite_vehicle_geometry_results.zip'
try:
    if not SCRIPT.is_file():raise RuntimeError('Helper missing; package readable logs directly')
    stage('bundle')
except Exception as error:
    print('Bundle helper unavailable:',error,'— packaging readable results/logs directly.')
    with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file() and 'assets' not in path.relative_to(OUTPUT).parts and path.suffix in ('.json','.csv','.txt','.log'):
                archive.write(path,path.relative_to(OUTPUT))
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
